# Conversation Derailment Forecasting — Chain-of-Thought & Social-Signal Prompting

**Individual contribution (Person 2)** to a group project comparing Zero-shot prompting, CoT / social-signal prompting, LoRA and QLoRA.

**Research question:** Without updating any model weights, does asking the LLM to (a) attend to social signals and/or (b) reason step by step before answering improve its forecast of whether a Reddit conversation will derail into a personal attack?

**Shared group protocol (identical to `cga_derailment_pipeline.py`):**

| Item | Setting |
|---|---|
| Model | `Qwen/Qwen2.5-1.5B-Instruct` (FP16, no fine-tuning) |
| Data | ConvoKit `conversations-gone-awry-cmv-corpus`, official split (train 4,106 / val 1,368 / test 1,368) |
| Input | all comments **except the last** (`all_but_last`), max 600 chars per comment, anonymised speakers |
| Label | `has_removed_comment` (final comment removed by CMV moderators under Rule 2) |
| Score | relative probability of `yes` vs `no` → risk score in [0, 1] |
| Evaluation | threshold chosen on **validation** accuracy from {0.05, …, 0.95}, then fixed for **test** |

**Prompt variants (the only thing that changes):**

| Variant | System prompt | Output |
|---|---|---|
| `A_base` | group `SYSTEM_PROMPT` (same as zero-shot and LoRA) | direct yes/no |
| `B_social` | task + social-signal checklist | direct yes/no |
| `C_cot` | task + "reason step by step" | short reasoning → `Answer: yes/no` |
| `D_social_cot` | task + social-signal checklist + reason through the signals | short reasoning → `Answer: yes/no` |
| `E_fewshot_cot` | same as D + 4 worked examples from the **train** split (section 13) | short reasoning → `Answer: yes/no` |

**How to run:** Runtime → Change runtime type → **T4 GPU**. Run the cells from top to bottom. All results are saved to Google Drive (`MyDrive/CoT_Project/`). If Colab disconnects, re-run sections 1–6 and then the cell you were on: it **resumes** where it stopped.

## 1 — Install packages (once per fresh runtime)
If the next cell shows an import error after installing, use *Runtime → Restart session* and continue from section 2.

In [1]:
!pip install -q convokit transformers accelerate scikit-learn pandas

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2 — Settings and Google Drive

In [3]:
import os, re, random
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from convokit import Corpus, download
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

CORPUS_NAME = "conversations-gone-awry-cmv-corpus"
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_CHARS = 600            # same as group pipeline
MAX_PROMPT_TOKENS = 1024   # same as group pipeline
MAX_NEW_TOKENS = 256       # length cap for CoT reasoning
DIRECT_BATCH = 8           # batch size for direct yes/no scoring
COT_BATCH = 16             # batch size for CoT generation (lower to 8 if you get "CUDA out of memory")
SEED = 42

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), "Select a GPU: Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/CoT_Project"
os.makedirs(SAVE_DIR, exist_ok=True)
print("Results will be saved to:", SAVE_DIR)

GPU: Tesla T4
Mounted at /content/drive
Results will be saved to: /content/drive/MyDrive/CoT_Project


## 3 — Data preparation (identical to the group pipeline)

The final comment is **never** shown to the model. Paired conversations stay in the same split (checked below). The asserted split sizes guarantee that we evaluate on exactly the same 1,368 test conversations as the other group members.

`build_context` also supports an integer `horizon` (first *k* comments) for the optional early-warning experiment; as in the LoRA notebook, it never includes the final comment even for very short conversations.

In [4]:
corpus = Corpus(filename=download(CORPUS_NAME))

def get_label(convo):
    if "conversation_has_personal_attack" in convo.meta:      # Wikipedia version
        return int(convo.meta["conversation_has_personal_attack"])
    return int(convo.meta["has_removed_comment"])             # CMV version

def build_context(convo, horizon="all_but_last"):
    utts = [u for u in convo.get_chronological_utterance_list()
            if not u.meta.get("is_section_header", False)]
    if horizon == "all_but_last":
        utts = utts[:-1]                                # never show the final comment
    else:
        utts = utts[:min(horizon, len(utts) - 1)]       # early warning, still never the final comment
    speakers, lines = {}, []
    for u in utts:
        name = speakers.setdefault(u.speaker.id, f"Speaker {len(speakers) + 1}")
        text = " ".join(u.text.split())[:MAX_CHARS]
        lines.append(f"{name}: {text}")
    return "\n".join(lines)

rows = [{"conv_id": c.id, "pair_id": c.meta.get("pair_id"), "split": c.meta.get("split"),
         "context": build_context(c), "label": get_label(c)}
        for c in corpus.iter_conversations()]
df = pd.DataFrame(rows)
df = df[df.context.str.len() > 0].copy()

split_of = dict(zip(df.conv_id, df.split))
leaks = sum(1 for p, s in zip(df.pair_id, df.split) if p in split_of and split_of[p] != s)
assert leaks == 0, "Paired conversations crossed splits!"

train_df = df[df.split == "train"].copy()
val_df = df[df.split == "val"].copy()
test_df = df[df.split == "test"].copy()
assert (len(train_df), len(val_df), len(test_df)) == (4106, 1368, 1368), "Split differs from the group's!"
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s} n={len(d):5d}  derail rate={d.label.mean():.2f}")

No configuration file found at /root/.convokit/config.yml; writing with contents: 
# Default Backend Parameters
db_host: localhost:27017
data_directory: ~/.convokit/saved-corpora
model_directory: ~/.convokit/saved-models
default_backend: mem
train n= 4106  derail rate=0.50
val   n= 1368  derail rate=0.50
test  n= 1368  derail rate=0.50


## 4 — Load the base model (no fine-tuning)

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"            # needed for batched generation
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16, device_map="auto")
model.eval()
DEVICE = next(model.parameters()).device

# Direct scoring: exactly the group's token ids
YES_ID = tokenizer.encode("yes", add_special_tokens=False)[0]
NO_ID = tokenizer.encode("no", add_special_tokens=False)[0]

# CoT scoring: after "Answer:" the model may write " yes", "Yes", ... -> pool the spelling variants
def first_token_ids(words):
    return sorted({tokenizer.encode(w, add_special_tokens=False)[0] for w in words})
YES_IDS = first_token_ids(["yes", " yes", "Yes", " Yes"])
NO_IDS = first_token_ids(["no", " no", "No", " No"])
assert not set(YES_IDS) & set(NO_IDS)
print("Model loaded on", DEVICE)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded on cuda:0


## 5 — Prompt variants

`BASE_TASK + " Answer only 'yes' or 'no'."` is **character-for-character the group's `SYSTEM_PROMPT`**, so `A_base` is directly comparable with the zero-shot and LoRA results. The other variants only change the system prompt; the user message is always the same.

The social-signal checklist is motivated by Zhang et al. (2018), who found that early conversational cues (direct questions, second-person framing, lack of politeness markers such as hedges or gratitude) predict later derailment.

**Edit only this cell when iterating on prompts.** If you change a prompt after running it, also rename the variant (e.g. `D_social_cot_v2`), otherwise the resume logic will reuse old saved results.

In [6]:
BASE_TASK = ("You are a moderation assistant for an online discussion forum. "
             "Read the conversation so far and forecast whether the NEXT comment will "
             "contain a personal attack (the conversation derails).")
SYSTEM_PROMPT = BASE_TASK + " Answer only 'yes' or 'no'."   # == group pipeline

SOCIAL_CUES = ("Pay attention to social signals: hostility or rude language; criticism aimed at the person "
               "rather than the argument; sarcasm or mocking; direct, confrontational questions and "
               "accusatory 'you' statements; tension that escalates from turn to turn. Politeness signals "
               "such as hedging ('I think', 'maybe'), acknowledging the other side ('fair point') or "
               "gratitude lower the risk. Strong disagreement alone is not a personal attack.")

COT_INSTRUCTION = ("First, reason step by step in at most 5 short sentences about how the conversation "
                   "is developing. Then write the final line exactly as 'Answer: yes' or 'Answer: no'.")

SOCIAL_COT_INSTRUCTION = ("First, go through these social signals in at most 5 short sentences, citing the "
                          "speakers' words where relevant. Then write the final line exactly as "
                          "'Answer: yes' or 'Answer: no'.")

VARIANTS = {
    "A_base":       {"system": SYSTEM_PROMPT, "mode": "direct"},
    "B_social":     {"system": BASE_TASK + " " + SOCIAL_CUES + " Answer only 'yes' or 'no'.", "mode": "direct"},
    "C_cot":        {"system": BASE_TASK + " " + COT_INSTRUCTION, "mode": "cot"},
    "D_social_cot": {"system": BASE_TASK + " " + SOCIAL_CUES + " " + SOCIAL_COT_INSTRUCTION, "mode": "cot"},
}

def build_messages(system, context, shots=None):
    # shots (few-shot variant only): earlier example turns, each answered in the CoT format
    msgs = [{"role": "system", "content": system}]
    for s in (shots or []):
        msgs.append({"role": "user", "content": f"Conversation so far:\n{s['context']}\n\nWill this conversation derail?"})
        msgs.append({"role": "assistant", "content": f"Reasoning: {s['rationale']}\nAnswer: {'yes' if s['label'] == 1 else 'no'}"})
    msgs.append({"role": "user", "content": f"Conversation so far:\n{context}\n\nWill this conversation derail?"})
    return msgs

def n_tokens(text):
    return len(tokenizer(text, add_special_tokens=False).input_ids)

def prompt_text(system, context, shots=None):
    # Chat-formatted prompt. If too long, drop the OLDEST comments (keep the end), so the
    # assistant turn is never cut off. Few-shot examples get extra room on top of
    # MAX_PROMPT_TOKENS, so the conversation being judged has the same budget in every variant.
    # Returns (text, was_truncated).
    limit = MAX_PROMPT_TOKENS
    if shots:
        with_shots = tokenizer.apply_chat_template(build_messages(system, "", shots), tokenize=False, add_generation_prompt=True)
        without = tokenizer.apply_chat_template(build_messages(system, ""), tokenize=False, add_generation_prompt=True)
        limit += n_tokens(with_shots) - n_tokens(without)
    lines, truncated = context.split("\n"), False
    while True:
        text = tokenizer.apply_chat_template(build_messages(system, "\n".join(lines), shots),
                                             tokenize=False, add_generation_prompt=True)
        if n_tokens(text) <= limit or len(lines) <= 1:
            return text, truncated
        lines, truncated = lines[1:], True

for k, v in VARIANTS.items():
    print(f"--- {k} ({v['mode']}) ---\n{v['system']}\n")

--- A_base (direct) ---
You are a moderation assistant for an online discussion forum. Read the conversation so far and forecast whether the NEXT comment will contain a personal attack (the conversation derails). Answer only 'yes' or 'no'.

--- B_social (direct) ---
You are a moderation assistant for an online discussion forum. Read the conversation so far and forecast whether the NEXT comment will contain a personal attack (the conversation derails). Pay attention to social signals: hostility or rude language; criticism aimed at the person rather than the argument; sarcasm or mocking; direct, confrontational questions and accusatory 'you' statements; tension that escalates from turn to turn. Politeness signals such as hedging ('I think', 'maybe'), acknowledging the other side ('fair point') or gratitude lower the risk. Strong disagreement alone is not a personal attack. Answer only 'yes' or 'no'.

--- C_cot (cot) ---
You are a moderation assistant for an online discussion forum. Read 

## 6 — Scoring functions

* **Direct** (`A`, `B`): one forward pass; score = softmax over the logits of `yes` / `no` (the group's method).
* **CoT** (`C`, `D`): (1) the assistant turn is pre-filled with `Reasoning:` (without it, Qwen tended to write the answer first, leaving no reasoning before `Answer:`), then the reasoning is generated greedily; (2) the reasoning is kept, everything from `Answer:` on is removed, and `\nAnswer:` is appended; (3) score = relative probability of yes vs no for the next token. This turns CoT into a continuous risk score, so the **same threshold protocol and ROC-AUC** apply. The text answer the model wrote is also stored (`parsed_answer`).

The score is a relative two-token probability, not a calibrated probability of derailment.

In [7]:
@torch.no_grad()
def last_token_logits(texts):
    enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
    pos = (enc.attention_mask.cumsum(-1) - 1).clamp(min=0)   # correct positions with left padding
    hidden = model.model(input_ids=enc.input_ids, attention_mask=enc.attention_mask,
                         position_ids=pos).last_hidden_state[:, -1, :]
    return model.lm_head(hidden).float()                      # only the last position: saves memory

def score_direct(texts):
    logits = last_token_logits(texts)
    return torch.softmax(logits[:, [YES_ID, NO_ID]], dim=-1)[:, 0].tolist()

ANSWER_START = re.compile(r"(final\s+)?answer\s*[:：]", re.I)
ANSWER_VALUE = re.compile(r"answer\s*[:：]\s*\**\s*(yes|no)", re.I)

@torch.no_grad()
def generate(texts):
    enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                         temperature=None, top_p=None, top_k=None,
                         pad_token_id=tokenizer.pad_token_id)
    return tokenizer.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)

PREFILL = "Reasoning:"   # the assistant turn starts with this, so the model must reason BEFORE answering

def score_cot(texts):
    gens = generate([t + PREFILL for t in texts])
    reasonings, parsed, followups = [], [], []
    for text, g in zip(texts, gens):
        m = ANSWER_START.search(g)
        reasoning = (g[:m.start()] if m else g).strip()
        a = ANSWER_VALUE.search(g)
        parsed.append(a.group(1).lower() if a else None)
        reasonings.append(reasoning)
        followups.append(text + PREFILL + " " + reasoning + "\nAnswer:")
    logits = last_token_logits(followups)
    diff = torch.logsumexp(logits[:, YES_IDS], -1) - torch.logsumexp(logits[:, NO_IDS], -1)
    return torch.sigmoid(diff).tolist(), reasonings, parsed, gens

def score_texts(variant, texts):
    if VARIANTS[variant]["mode"] == "direct":
        n = len(texts)
        return score_direct(texts), [""] * n, [None] * n, [""] * n
    return score_cot(texts)

## 7 — Run / resume / evaluate helpers

Each (variant, split) is saved batch by batch to `MyDrive/CoT_Project/<variant>_<tag>.csv`. Re-running skips conversations that are already saved, so a Colab disconnect costs at most one batch.

In [8]:
def run_variant(variant, data, tag):
    path = f"{SAVE_DIR}/{variant}_{tag}.csv"
    done_ids = set(pd.read_csv(path)["conv_id"]) if os.path.exists(path) else set()
    todo = data[~data.conv_id.isin(done_ids)]
    if len(todo):
        system = VARIANTS[variant]["system"]
        items = []
        for cid, ctx, y in zip(todo.conv_id, todo.context, todo.label):
            text, trunc = prompt_text(system, ctx, VARIANTS[variant].get("shots"))
            items.append((cid, int(y), text, trunc))
        items.sort(key=lambda x: -len(x[2]))      # longest first: memory problems show up immediately
        bs = VARIANTS[variant].get("batch") or (COT_BATCH if VARIANTS[variant]["mode"] == "cot" else DIRECT_BATCH)
        for i in tqdm(range(0, len(items), bs), desc=f"{variant} [{tag}]"):
            batch = items[i:i + bs]
            probs, reasonings, parsed, raws = score_texts(variant, [b[2] for b in batch])
            pd.DataFrame({"conv_id": [b[0] for b in batch], "label": [b[1] for b in batch],
                          "prob": probs, "truncated": [b[3] for b in batch],
                          "parsed_answer": parsed, "reasoning": reasonings, "raw_output": raws}
                         ).to_csv(path, mode="a", header=not os.path.exists(path), index=False)
    res = pd.read_csv(path)
    for col in ["reasoning", "raw_output"]:          # tolerate files saved by an older notebook version
        res[col] = res[col].fillna("") if col in res else ""
    return res[res.conv_id.isin(set(data.conv_id))].reset_index(drop=True)

def pick_threshold(val_res):
    # Same rule as the group: maximise validation accuracy over {0.05, ..., 0.95}
    return max([i / 20 for i in range(1, 20)],
               key=lambda t: accuracy_score(val_res.label, (val_res.prob >= t).astype(int)))

def evaluate(res, threshold, variant):
    y, pred = res.label, (res.prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    out = {"variant": variant, "n": len(res), "threshold": threshold,
           "accuracy": accuracy_score(y, pred), "precision": precision_score(y, pred, zero_division=0),
           "recall": recall_score(y, pred), "f1": f1_score(y, pred), "roc_auc": roc_auc_score(y, res.prob),
           "TN": tn, "FP": fp, "FN": fn, "TP": tp, "truncated": int(res.truncated.sum())}
    if VARIANTS[variant]["mode"] == "cot":
        ok = res.parsed_answer.isin(["yes", "no"])
        out["answer_line_rate"] = ok.mean()                    # did the model follow the format?
        out["text_answer_accuracy"] = ((res.parsed_answer[ok] == "yes").astype(int) == res.label[ok]).mean()
    return out

## 8 — Development run: tune prompts on 100 validation conversations

Use this small sample to check that everything works and to **iterate on the prompts** (section 5). Read the reasoning: does the model follow the format? Does it mention the social signals? Never tune prompts on the test set.

Rough time on a T4: direct variants take about a minute; CoT variants take several minutes for 100 conversations.

In [18]:
dev_df = val_df.sample(n=100, random_state=SEED)
dev_rows = []
for v in VARIANTS:
    res = run_variant(v, dev_df, "dev")
    r = evaluate(res, 0.5, v)          # fixed 0.5 only for this quick check
    dev_rows.append(r)
pd.DataFrame(dev_rows)[["variant", "accuracy", "f1", "roc_auc"] +
                       [c for c in ["answer_line_rate"] if c in pd.DataFrame(dev_rows)]]

,variant,accuracy,f1,roc_auc,answer_line_rate
0,A_base,0.56,0.645161,0.563502,NaN
1,B_social,0.56,0.666667,0.546675,NaN
2,C_cot,0.56,0.435897,0.627003,1.00
3,D_social_cot,0.60,0.615385,0.618590,1.00
4,E_fewshot_cot,0.58,0.650000,0.554287,0.99


In [19]:
# Read a few reasoning examples
res = run_variant("D_social_cot", dev_df, "dev")
for _, r in res.head(3).iterrows():
    print(f"=== {r.conv_id} | actual={r.label} | score={r.prob:.3f} | text answer={r.parsed_answer}")
    print("REASONING:", r.reasoning)
    print("RAW OUTPUT:", r.raw_output, "\n")

=== dh8xnbk | actual=0 | score=0.985 | text answer=yes
REASONING: - Speaker 3 criticizes the electoral college as unfairly favoring certain ideologies.
- Speaker 4 argues that the electoral college represents the interests of less populous states.
- Speaker 3 responds by questioning why populous states have more representation.

Final
RAW OUTPUT:  
- Speaker 3 criticizes the electoral college as unfairly favoring certain ideologies.
- Speaker 4 argues that the electoral college represents the interests of less populous states.
- Speaker 3 responds by questioning why populous states have more representation.

Final answer: Answer: yes 

=== d9xte07 | actual=0 | score=0.012 | text answer=no
REASONING: Speaker 2 responds directly to Speaker 1's claim that the law applies to private actors, stating that the majority of constitutional protections only apply as between the government and the people, not as between private actors. This response maintains the focus on the application of the la

## 9 — Full run: validation (choose threshold) → test (report once)

Only run this after the prompts are final. For each variant: score all 1,368 validation conversations, pick the threshold, then score the 1,368 test conversations with that threshold fixed.

Rough time on a T4: direct variants take minutes; each CoT variant may take one to a few hours for val + test. If Colab disconnects, re-run sections 1–7 and this cell; it resumes. You can also run one variant at a time by editing `RUN`.

In [20]:
RUN = ["A_base", "B_social", "C_cot", "D_social_cot"]

summary, thresholds = [], {}
for v in RUN:
    val_res = run_variant(v, val_df, "val")
    thresholds[v] = pick_threshold(val_res)
    test_res = run_variant(v, test_df, "test")
    summary.append(evaluate(test_res, thresholds[v], v))

summary_df = pd.DataFrame(summary)
summary_df.to_csv(f"{SAVE_DIR}/results_summary_test.csv", index=False)
summary_df.round(4)

,variant,n,threshold,accuracy,precision,recall,f1,roc_auc,TN,FP,FN,TP,truncated,answer_line_rate,text_answer_accuracy
0,A_base,1368,0.85,0.5746,0.5977,0.4561,0.5174,0.6030,474,210,372,312,100,NaN,NaN
1,B_social,1368,0.85,0.5950,0.5923,0.6096,0.6009,0.6187,397,287,267,417,126,NaN,NaN
2,C_cot,1368,0.15,0.5680,0.5823,0.4810,0.5268,0.5924,448,236,355,329,108,1.0,0.5621
3,D_social_cot,1368,0.95,0.5994,0.6214,0.5088,0.5595,0.6113,472,212,336,348,134,1.0,0.5885


## 10 — Error analysis: where does CoT change the decision?

Compares the base prompt (`A_base`) with the social-signal CoT prompt (`D_social_cot`) on the **whole** test set (not hand-picked cases): how many errors were fixed, how many correct answers were broken, and example reasoning for each.

In [21]:
BASE, NEW = "A_base", "D_social_cot"
a = run_variant(BASE, test_df, "test")[["conv_id", "label", "prob"]].rename(columns={"prob": "p_base"})
d = run_variant(NEW, test_df, "test")[["conv_id", "prob", "reasoning"]].rename(columns={"prob": "p_new"})
cmp = a.merge(d, on="conv_id")
cmp["pred_base"] = (cmp.p_base >= thresholds[BASE]).astype(int)
cmp["pred_new"] = (cmp.p_new >= thresholds[NEW]).astype(int)
fixed = cmp[(cmp.pred_base != cmp.label) & (cmp.pred_new == cmp.label)]
broken = cmp[(cmp.pred_base == cmp.label) & (cmp.pred_new != cmp.label)]
print(f"Decisions changed: {(cmp.pred_base != cmp.pred_new).sum()} / {len(cmp)}")
print(f"Errors fixed by {NEW}: {len(fixed)} | correct answers broken: {len(broken)}")
cmp.to_csv(f"{SAVE_DIR}/error_analysis_{BASE}_vs_{NEW}.csv", index=False)

for title, part in [("FIXED", fixed), ("BROKEN", broken)]:
    for _, r in part.head(2).iterrows():
        print(f"\n=== {title} | {r.conv_id} | actual={r.label} | base={r.p_base:.3f} new={r.p_new:.3f}")
        print("Context:\n" + test_df.set_index("conv_id").context[r.conv_id][:800])
        print("Reasoning:\n" + str(r.reasoning))

Decisions changed: 446 / 1368
Errors fixed by D_social_cot: 240 | correct answers broken: 206

=== FIXED | dd99rp2 | actual=1 | base=0.725 new=1.000
Context:
Speaker 1: 1) There were 2 conflicting protests at UC Berkeley. There was a peaceful protest of 1,500 people and a group of 150 violent black bloc protesters. When a large number of people gather, there's always a risk of violence, even if the protest is explicitly peaceful. Should the protest organizers be held responsible for the actions of an unaffiliated minority? 2) Milo would have a massive platform for his views even if this protest didn't happen. His views are being espoused by the President of the United States. You can't get a bigger platform than that.
Speaker 2: &gt;2) Milo would have a massive platform for his views even if this protest didn't happen. His views are being espoused by the President of the United States. You can't get a bigger platform than that. So as long as so
Reasoning:
Speaker 2 accuses Speaker 1 of

## 11 — User scenarios and crash test (shared with the group)

The same three hand-written conversations as in `cga_derailment_pipeline.py`, so that all four methods can be contrasted on identical inputs. Feel free to add more scenarios. The CoT reasoning shows *why* the model succeeds or fails.

In [25]:
scenarios = {
    "Scenario 1 - polite disagreement":
        "Speaker 1: I think the article overstates the risks.\n"
        "Speaker 2: Fair point, though the 2023 data does support it.",
    "Scenario 2 - escalating tension":
        "Speaker 1: Did you even read the source?\n"
        "Speaker 2: Yes, unlike you apparently.\n"
        "Speaker 1: Wow. Typical.",
    "Crash test - sarcasm / friendly banter":
        "Speaker 1: You absolute genius, you broke the build again :)\n"
        "Speaker 2: Haha guilty, buying coffee tomorrow.",
}

thr = globals().get("thresholds", {})
scen_rows = []
for name, ctx in scenarios.items():
    print(f"\n################ {name}")
    for v in VARIANTS:
        text, _ = prompt_text(VARIANTS[v]["system"], ctx, VARIANTS[v].get("shots"))
        probs, reasonings, parsed, _ = score_texts(v, [text])
        t = thr.get(v, 0.5)
        print(f"{v:13s} score={probs[0]:.3f}  flag={int(probs[0] >= t)} (threshold {t})")
        if reasonings[0]:
            print("   reasoning:", reasonings[0].replace("\n", " "))
        scen_rows.append({"scenario": name, "variant": v, "score": probs[0], "threshold": t,
                          "flag": int(probs[0] >= t), "reasoning": reasonings[0]})
pd.DataFrame(scen_rows).to_csv(f"{SAVE_DIR}/scenario_results.csv", index=False)


################ Scenario 1 - polite disagreement
A_base        score=0.049  flag=0 (threshold 0.85)
B_social      score=0.034  flag=0 (threshold 0.85)
C_cot         score=0.000  flag=0 (threshold 0.15)
   reasoning: - Speaker 1 expresses agreement with the article's stance on risks. - Speaker 2 acknowledges that while they agree with the article, the 2023 data supports their view. - The dialogue appears to be discussing supporting evidence rather than escalating into a heated argument.
D_social_cot  score=0.000  flag=0 (threshold 0.95)
   reasoning: Speaker 2 acknowledges the fairness of Speaker 1's view by referencing recent data, which suggests they are not directly attacking Speaker 1 personally but rather addressing their argument with evidence.
E_fewshot_cot score=0.000  flag=0 (threshold 0.8)
   reasoning: Speaker 2 acknowledges the fairness of the previous statement and provides additional context ('Fair point, though the 2023 data does support it'). This maintains the focus o

## 12 — Optional: early warning (first 1, 2, 3 comments)

Same idea as the LoRA notebook: score the same test conversations using only the first *k* comments, with the full-context validation threshold kept fixed. Direct variants are cheap; CoT variants take much longer.

In [23]:
EW_VARIANTS = ["A_base", "B_social"]     # add "D_social_cot" if you have time
ew_rows = []
for v in EW_VARIANTS:
    for h in [1, 2, 3]:
        ew = test_df.copy()
        ew["context"] = [build_context(corpus.get_conversation(cid), horizon=h) for cid in ew.conv_id]
        res = run_variant(v, ew, f"test_h{h}")
        r = evaluate(res, thresholds[v], v); r["horizon"] = h
        ew_rows.append(r)
ew_df = pd.DataFrame(ew_rows)[["variant", "horizon", "accuracy", "f1", "roc_auc"]]
ew_df.to_csv(f"{SAVE_DIR}/early_warning_results_cot.csv", index=False)
ew_df.round(4)

A_base [test_h1]:   0%|          | 0/171 [00:00<?, ?it/s]

A_base [test_h2]:   0%|          | 0/171 [00:00<?, ?it/s]

A_base [test_h3]:   0%|          | 0/171 [00:00<?, ?it/s]

B_social [test_h1]:   0%|          | 0/171 [00:00<?, ?it/s]

B_social [test_h2]:   0%|          | 0/171 [00:00<?, ?it/s]

B_social [test_h3]:   0%|          | 0/171 [00:00<?, ?it/s]

,variant,horizon,accuracy,f1,roc_auc
0,A_base,1,0.5015,0.0658,0.5282
1,A_base,2,0.5124,0.2412,0.5450
2,A_base,3,0.5292,0.3586,0.5592
3,B_social,1,0.5066,0.1466,0.5349
4,B_social,2,0.5285,0.3944,0.5604
5,B_social,3,0.5534,0.4809,0.5848


## 13 — Few-shot CoT (variant E): prompting with training data

**Idea:** the same system prompt as `D_social_cot`, plus **4 worked examples taken from the training split** (2 that derailed, 2 that stayed civil), shown as earlier turns of the chat. Each example has a short reasoning that walks through the social signals, then `Answer: yes/no`. This uses the application's data to "train" the model through the prompt, without changing any weights.

**How the example reasoning is made:** the model itself writes it, *given the correct answer* (label-conditioned rationales, as in STaR, Zelikman et al., 2022). The rationales are saved to Drive (`fewshot_examples.json`) so the prompt is fixed, and you should **read and, if needed, correct them before running E**.

**Fairness:** examples come only from the train split, never from val or test. The conversation being judged gets the same token budget as in A–D; the examples get extra room on top.

Run sections 1–7 first (the A–D results already saved on Drive are reused, not recomputed).

In [9]:
import json
FEWSHOT_PATH = f"{SAVE_DIR}/fewshot_examples.json"

# 13.1 Pick 4 short training conversations: order yes, no, no, yes; never two from the same pair
pool = train_df[(train_df.context.str.len() <= 450) & (train_df.context.str.count("\n") >= 1)]
print("Short training conversations available:", len(pool), "| by label:", pool.label.value_counts().to_dict())

if os.path.exists(FEWSHOT_PATH):
    shots = json.load(open(FEWSHOT_PATH))
    print("Loaded saved examples from Drive (delete the file to pick new ones).")
else:
    rng = np.random.RandomState(SEED)
    shots, used = [], set()
    for lab in [1, 0, 0, 1]:
        cand = pool[(pool.label == lab) & ~pool.conv_id.isin(used) & ~pool.pair_id.isin(used)]
        r = cand.sample(1, random_state=rng).iloc[0]
        used |= {r.conv_id, r.pair_id}
        shots.append({"conv_id": r.conv_id, "label": int(r.label), "context": r.context, "rationale": ""})

Short training conversations available: 136 | by label: {0: 73, 1: 63}


In [10]:
# 13.2 Label-conditioned rationales, written by the model and then reviewed by you
RATIONALE_SYSTEM = (BASE_TASK + " " + SOCIAL_CUES + " You will be told the correct answer. In 2 to 4 short "
                    "sentences, explain which social signals in the conversation support that answer, quoting "
                    "the speakers' words where relevant. Do not mention that you were told the answer, and do "
                    "not write an 'Answer:' line.")

def gen_rationale(ctx, label):
    told = ("yes: the next comment was removed by moderators as rude or hostile" if label == 1
            else "no: the conversation stayed civil")
    msgs = [{"role": "system", "content": RATIONALE_SYSTEM},
            {"role": "user", "content": f"Conversation so far:\n{ctx}\n\nCorrect answer: {told}. Explain why."}]
    out = generate([tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)])[0]
    m = ANSWER_START.search(out)
    out = (out[:m.start()] if m else out).strip()
    return re.sub(r"^\s*reasoning\s*:\s*", "", out, flags=re.I).strip()

for s in shots:
    if not s["rationale"]:
        s["rationale"] = gen_rationale(s["context"], s["label"])
json.dump(shots, open(FEWSHOT_PATH, "w"), ensure_ascii=False, indent=1)

for i, s in enumerate(shots, 1):
    print(f"===== Example {i} | {s['conv_id']} | label={s['label']} =====")
    print(s["context"])
    print("REASONING:", s["rationale"], "\n")

===== Example 1 | e7rrl4u | label=1 =====
Speaker 1: Are you really open to changing your mind, because you sound downright hostile?
Speaker 2: Excuse me if i sound hostile, I didn't mean it... I do get angry talking about this because I feel so strongly. I don't think anything anyone in the world can say anything to change my mind on here but hence the reason I posted it .... here in change my view. Keyword: think
REASONING: The conversation has escalated into a heated debate with Speaker 1 questioning Speaker 2's tone of hostility and Speaker 2 responding defensively. The use of phrases like "hostile" and "angry" indicates a negative emotional state, suggesting a personal attack. Speaker 2 also uses capitalization ("here") and punctuation ("keyword"), which can sometimes indicate frustration or anger. These social signals align with the criteria for identifying a personal attack, making it likely that the next comment would be removed by moderators as rude or hostile. 

===== Example

In [14]:
# 13.3 OPTIONAL: correct a rationale by hand. Put {conv_id: "new text"} below and run this cell.
# If you change an example AFTER running E, also rename the variant (e.g. E_fewshot_cot_v2).
EDITS = {
    "e7rrl4u": "Speaker 1 opens with an accusatory 'you' question about Speaker 2's attitude rather than the argument: 'you sound downright hostile'. Speaker 2 apologises but admits 'I do get angry' and says nothing anyone can say will change their mind. The tension is now personal and there is no common ground, so the next comment is likely to turn into an attack.",
    "dv17vh9": "The exchange stays on the topic, not the people. 'Fragile snowflake' is aimed at people in general, not at the other speaker, and is softened by 'I don't think'. Speaker 2 asks a genuine question ('Can you show me a liberal subreddit...'), and Speaker 3 simply answers it with an example. There is no sarcasm or personal criticism, so the risk is low.",
    "dtou1pj": "Both speakers argue about ideas: Speaker 1 offers a hypothetical ('What if...'), and Speaker 2 disagrees with the premise, not with the person. There are no insults, no sarcasm and no accusatory 'you' statements, and the tension does not rise between turns. This is strong disagreement, which alone is not a personal attack.",
    "e770svk": "The replies get shorter and more curt with each turn ('Not if you over do it', 'Not as bad as Psilocybin'), a sign of rising tension. Speaker 2 brushes off a warning with 'Give it a try', and neither speaker acknowledges the other's point. They are repeating their positions instead of engaging, so the exchange is likely to turn personal.",
}
for s in shots:
    if s["conv_id"] in EDITS:
        s["rationale"] = EDITS[s["conv_id"]].strip()
json.dump(shots, open(FEWSHOT_PATH, "w"), ensure_ascii=False, indent=1)

VARIANTS["E_fewshot_cot"] = {"system": VARIANTS["D_social_cot"]["system"], "mode": "cot", "shots": shots, "batch": 8}
example_text, _ = prompt_text(VARIANTS["E_fewshot_cot"]["system"], test_df.context.iloc[0], shots)
print("Few-shot prompt length for one test conversation:", n_tokens(example_text), "tokens")

Few-shot prompt length for one test conversation: 1299 tokens


In [15]:
# 13.4 Quick check on the same 100 dev conversations (fixed 0.5 threshold, as in section 8)
dev_df = val_df.sample(n=100, random_state=SEED)   # same sample as section 8
res = run_variant("E_fewshot_cot", dev_df, "dev")
print({k: round(v, 3) if isinstance(v, float) else v for k, v in evaluate(res, 0.5, "E_fewshot_cot").items()
       if k in ["accuracy", "f1", "roc_auc", "answer_line_rate"]})
for _, r in res.head(2).iterrows():
    print(f"\n=== {r.conv_id} | actual={r.label} | score={r.prob:.3f}\n{r.reasoning}")

{'accuracy': 0.58, 'f1': 0.65, 'roc_auc': np.float64(0.554), 'answer_line_rate': np.float64(0.99)}

=== dh8xnbk | actual=0 | score=0.872
The conversation has become increasingly contentious and personal, with both sides presenting strong arguments and counterarguments. Speaker 3 continues to defend the Electoral College system, arguing that it ensures representation regardless of ideological leanings. Speaker 4 counters by highlighting instances where the Electoral College has favored candidates contrary to popular vote outcomes, implying that it unfairly represents the interests of larger populations. Both speakers engage in lengthy, detailed explanations that often involve quoting historical events and legal principles. While the discourse remains centered around political systems and voting mechanisms, the intensity of the debate suggests that it may escalate further into personal attacks or extreme positions. Given the complexity of the topic and the depth of the arguments presente

In [24]:
# 13.5 Full run: threshold on val, then test once. A–D are read back from Drive, not recomputed.
thresholds = {v: pick_threshold(run_variant(v, val_df, "val")) for v in VARIANTS}
summary = [evaluate(run_variant(v, test_df, "test"), thresholds[v], v) for v in VARIANTS]
summary_df = pd.DataFrame(summary)
summary_df.to_csv(f"{SAVE_DIR}/results_summary_test_with_E.csv", index=False)
summary_df.round(4)

,variant,n,threshold,accuracy,precision,recall,f1,roc_auc,TN,FP,FN,TP,truncated,answer_line_rate,text_answer_accuracy
0,A_base,1368,0.85,0.5746,0.5977,0.4561,0.5174,0.6030,474,210,372,312,100,NaN,NaN
1,B_social,1368,0.85,0.5950,0.5923,0.6096,0.6009,0.6187,397,287,267,417,126,NaN,NaN
2,C_cot,1368,0.15,0.5680,0.5823,0.4810,0.5268,0.5924,448,236,355,329,108,1.0000,0.5621
3,D_social_cot,1368,0.95,0.5994,0.6214,0.5088,0.5595,0.6113,472,212,336,348,134,1.0000,0.5885
4,E_fewshot_cot,1368,0.80,0.5914,0.5703,0.7412,0.6446,0.6238,302,382,177,507,134,0.9942,0.5868


In [17]:
# 13.6 McNemar tests: does few-shot beat the base prompt (A) and the zero-shot CoT prompt (D)?
from scipy.stats import binomtest
def mcnemar(base, new):
    a = run_variant(base, test_df, "test").set_index("conv_id")
    b = run_variant(new, test_df, "test").set_index("conv_id").loc[a.index]
    pa = (a.prob >= thresholds[base]).astype(int)
    pb = (b.prob >= thresholds[new]).astype(int)
    fixed = int(((pa != a.label) & (pb == a.label)).sum())
    broken = int(((pa == a.label) & (pb != a.label)).sum())
    return fixed, broken, binomtest(fixed, fixed + broken, 0.5).pvalue
for base in ["A_base", "D_social_cot"]:
    f, b, p = mcnemar(base, "E_fewshot_cot")
    print(f"E_fewshot_cot vs {base}: fixed={f}, broken={b}, McNemar p={p:.3f}")

E_fewshot_cot vs A_base: fixed=269, broken=246, McNemar p=0.332
E_fewshot_cot vs D_social_cot: fixed=245, broken=256, McNemar p=0.655


## Limitations to report

* The label is a CMV-specific moderation outcome (Rule 2 removal), not a universal definition of a personal attack.
* `all_but_last` context can already contain hostility; this is forecasting the last comment, not necessarily warning before any harm.
* Yes/no scores are relative two-token probabilities, not calibrated probabilities.
* CoT reasoning is generated text: it can sound convincing while being unfaithful to what actually drives the score.
* Very long conversations are shortened by dropping the oldest comments (count reported in the `truncated` column).

## References

* Wei et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models.* NeurIPS.
* Kojima et al. (2022). *Large Language Models are Zero-Shot Reasoners.* NeurIPS.
* Zelikman et al. (2022). *STaR: Bootstrapping Reasoning With Reasoning.* NeurIPS.
* Zhang et al. (2018). *Conversations Gone Awry: Detecting Early Signs of Conversational Failure.* ACL.
* Chang & Danescu-Niculescu-Mizil (2019). *Trouble on the Horizon: Forecasting the Derailment of Online Conversations as they Develop.* EMNLP.
* Qwen Team (2024). *Qwen2.5 Technical Report.* arXiv:2412.15115.